In [2]:
import sys
sys.path.append('../src')
from model import SimpleCNN
from partition import dirichlet_partition
from federated import local_train, federated_average, evaluate

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

transform = transforms.ToTensor()
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
train_labels = [label for _, label in train_dataset]

num_clients = 5
num_rounds = 5   # number of communication rounds
alpha = 0.5

client_data = dirichlet_partition(train_labels, num_clients=num_clients, alpha=alpha)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

global_model = SimpleCNN(num_classes=10)

for round_num in range(num_rounds):
    client_state_dicts = []
    
    for client_id in range(num_clients):
        subset = Subset(train_dataset, client_data[client_id])
        dataloader = DataLoader(subset, batch_size=32, shuffle=True)
        state_dict = local_train(global_model, dataloader, epochs=1, lr=0.01)
        client_state_dicts.append(state_dict)
    
    new_global_state = federated_average(client_state_dicts)
    global_model.load_state_dict(new_global_state)
    
    acc = evaluate(global_model, test_loader)
    print(f"Round {round_num+1}/{num_rounds} - Global model test accuracy: {acc:.4f}")

Round 1/5 - Global model test accuracy: 0.1313
Round 2/5 - Global model test accuracy: 0.1975
Round 3/5 - Global model test accuracy: 0.2469
Round 4/5 - Global model test accuracy: 0.2568
Round 5/5 - Global model test accuracy: 0.2933
